# SpaCal-Depth: Depth Pro caching on Colab (third base model)

Runtime → Change runtime type → **A100** (L4 / T4: set `AMP = True`; T4 may run out of memory at Depth Pro's 1536 x 1536 input).

Before the first session: upload a fresh `spacal_code.tgz` (it must contain `src/models/depth_pro.py`) to `MyDrive/spacal/`.

Everything goes to separate Drive folders (`cache_depthpro`, `prepared_depthpro`, `results_depthpro`), so the earlier caches and E0 results are never touched. Resumable: after a disconnect, Run all again.

Output to bring back: `MyDrive/spacal/prepared_depthpro/*__depthpro.npz` (→ cluster `prepared/`) and `MyDrive/spacal/results_depthpro/e0/per_image.csv`.

In [ ]:
# 1. GPU + Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc && free -g | head -2 && df -h /content | tail -1
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Configuration
import os
DATASETS = ['nyu', 'kitti', 'diode_indoor', 'diode_outdoor']   # trim to resume selectively
AMP = False              # A100: full precision. L4 / T4: True
PRUNE_AFTER = False      # True deletes the raw Depth Pro cache once its table exists (keeps RGB samples)

DRIVE = '/content/drive/MyDrive/spacal'
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
os.environ.update({
    'SPACAL_DATA_ROOT': '/content/data',
    'SPACAL_CACHE_ROOT': f'{DRIVE}/cache_depthpro',
    'SPACAL_PREPARED_ROOT': f'{DRIVE}/prepared_depthpro',
    'SPACAL_RESULTS_ROOT': f'{DRIVE}/results_depthpro',
    'HF_HOME': '/content/hf_cache',
    'SPACAL_DEVICE': 'cuda',
    'PYTHONUNBUFFERED': '1',
})
DL_TARGETS = sorted({'nyu': 'nyu', 'kitti': 'kitti', 'diode_indoor': 'diode', 'diode_outdoor': 'diode'}[d] for d in DATASETS)
ENV = 'MODEL_JOBS="depthpro --flip" ' + ('AMP=1 ' if AMP else '')
print('datasets:', DATASETS, '| downloads:', DL_TARGETS, '| env:', ENV)

In [ ]:
# 3. Code
!rm -rf /content/CV_Project && tar xzf {DRIVE}/spacal_code.tgz -C /content
%cd /content/CV_Project
!test -f src/models/depth_pro.py && echo 'depth_pro.py present' || echo '!!! old tarball: upload a fresh spacal_code.tgz'

In [ ]:
# 4. Dependencies (Colab's torch is kept; transformers only upgraded if older than 4.49)
!pip install -q "transformers>=4.49" "huggingface_hub>=0.22"
!python -c "import torch, transformers; from transformers import DepthProForDepthEstimation; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, '| DepthPro ok')"

In [ ]:
# 5. Data (local disk, ~8 GB for all three sources, ~15-30 min)
!bash scripts/download_data.sh {' '.join(DL_TARGETS)}

In [ ]:
# 6. Smoke test (3 images per dataset) + scale sanity check.
# median(pred / gt) should be roughly 0.6-1.6; a value off by 10x means the FOV conversion went wrong: stop.
for ds in DATASETS:
    !{ENV} LIMIT=3 bash scripts/cache_dataset.sh {ds}
import numpy as np
from src.data.cache import list_samples, load_gt, load_pred
for ds in DATASETS:
    for sid in list_samples(ds, 'depthpro')[:3]:
        g, p = load_gt(ds, sid), load_pred(ds, 'depthpro', sid)
        m = g['mask'].astype(bool)
        print(f"{ds:14s} {sid[:40]:40s} median pred/gt = {np.median(p['pred'][m] / g['gt'][m]):.3f}  t_pred = {float(p['t_pred']):.2f}s")

In [ ]:
# 7. Full caching (hours; keep the tab open). Safe to re-run after a disconnect.
for ds in DATASETS:
    !{ENV} bash scripts/cache_dataset.sh {ds} 2>&1 | tee -a {DRIVE}/logs/cache_depthpro_{ds}.log

In [ ]:
# 8. Status: a dataset is COMPLETE when its Depth Pro manifest finished a full (non-smoke) run
import glob, json
complete = []
for ds in ['nyu', 'kitti', 'diode_indoor', 'diode_outdoor']:
    man = f"{os.environ['SPACAL_CACHE_ROOT']}/{ds}/depthpro/manifest.json"
    if not os.path.exists(man):
        print(f'{ds:14s} not started'); continue
    m = json.load(open(man))
    done = 'finished' in m and not m['args']['limit'] and m['n_cached'] >= m['n_samples']
    complete += [ds] if done else []
    print(f"{ds:14s} {m.get('n_cached', '?')}/{m['n_samples']}", 'COMPLETE' if done else '')
print('complete:', complete)
!du -sh {DRIVE}/cache_depthpro 2>/dev/null

In [ ]:
# 9. E0 metrics + PixelTables for completed datasets (+ optional prune)
ready = [d for d in complete if d in DATASETS]
if ready:
    !python -m experiments.e0_base_metrics --datasets {' '.join(ready)}
    for ds in ready:
        !python -m experiments.prepare_tables --dataset {ds} --model depthpro
        if PRUNE_AFTER:
            !python -m scripts.prune_cache --dataset {ds} --yes
else:
    print('nothing complete yet')
!ls -la {DRIVE}/prepared_depthpro

## After all four datasets are finished

**Disconnect the GPU runtime.** Copy `MyDrive/spacal/prepared_depthpro/*__depthpro.npz` into the cluster's `prepared/` and `results_depthpro/e0/per_image.csv` to the laptop, then on the cluster:

`sbatch -p phd --export=ALL,PARTS=depthpro slurm/new.sbatch`

(The table step sees the tables already exist and skips straight to the main study, E5, E11 and E12.)